In [6]:
import pandas as pd
df = pd.read_csv("dataset/synthetic_logs.csv")
df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [7]:
df.source.unique()

<StringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [8]:
df.target_label.unique()

<StringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
import numpy as np

#Load pre-trained SentenceTransformer Model
model = SentenceTransformer('all-MiniLM-L6-v2')

#generate embeddings for log messages
embeddings = model.encode(df['log_message'].tolist())
embeddings[:2]

array([[-1.02939658e-01,  3.35459076e-02, -2.20260452e-02,
         1.55100238e-03, -9.86919273e-03, -1.78956255e-01,
        -6.34410605e-02, -6.01761863e-02,  2.81108301e-02,
         5.99620342e-02, -1.72619056e-02,  1.43371313e-03,
        -1.49560124e-01,  3.15283332e-03, -5.66031076e-02,
         2.71685813e-02, -1.49890380e-02, -3.54037471e-02,
        -3.62936407e-02, -1.45410290e-02, -5.61491679e-03,
         8.75538141e-02,  4.55120839e-02,  2.50964109e-02,
         1.00187147e-02,  1.24267694e-02, -1.39923632e-01,
         7.68696144e-02,  3.14095356e-02, -4.15251637e-03,
         4.36902493e-02,  1.71250291e-02, -8.00951049e-02,
         5.74006177e-02,  1.89091284e-02,  8.55261385e-02,
         3.96398529e-02, -1.34371802e-01, -1.44369516e-03,
         3.06704547e-03,  1.76854089e-01,  4.44884971e-03,
        -1.69275030e-02,  2.24266127e-02, -4.35050204e-02,
         6.09026849e-03, -9.98166855e-03, -6.23973049e-02,
         1.07372068e-02, -6.04897272e-03, -7.14660585e-0

In [14]:
#Perform DBCAN clustering
dbscan = DBSCAN(eps=0.2, min_samples = 5, metric='cosine')
clusters = dbscan.fit_predict(embeddings)

#Add cluster labels to the dataframe
df['cluster'] = clusters
df.head()

,timestamp,source,log_message,target_label,complexity,cluster
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,-1
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0


In [15]:
df[df.cluster == 1].head()

,timestamp,source,log_message,target_label,complexity,cluster
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert,1
217,1/22/2025 5:45,BillingSystem,Mail service encountered a delivery glitch,Error,bert,1
248,5/2/2025 23:04,ModernHR,Service disruption caused by email sending error,Critical Error,bert,1
265,3/30/2025 23:53,ModernCRM,Email system had a problem sending emails,Error,bert,1


In [17]:
df['cluster'] = 5

In [19]:
cluster_counts = df['cluster'].value_counts()
large_clusters = cluster_counts[cluster_counts > 10].index

for clusters in large_clusters:
    print(f"Cluster {clusters}: ")
    print(df[df['cluster'] == clusters]['log_message'].head(5).to_string(index=False))
    print()

Cluster 5: 
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
    Email service experiencing issues with sending
         Unauthorized access to data was attempted
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...



In [27]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message, re.IGNORECASE):
            return label
        return None

In [32]:
classify_with_regex("User User494 logged in.")

'User Action'

In [36]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)
df[df.regex_label.isna()]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,5,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,5,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,5,NaN
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,5,NaN
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,5,NaN
...,...,...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,5,NaN
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,5,NaN
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,5,NaN
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert,5,NaN


In [38]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,5,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,5,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,5,NaN
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,5,NaN
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,5,NaN
...,...,...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,5,NaN
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,5,NaN
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,5,NaN
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert,5,NaN


In [ ]:
print(df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts() <= 5].index.tolist())

In [40]:
df_non_legacy = df_non_regex[df_non_regex.source!='LegacyCRM']
df_non_legacy.source.unique()

<StringArray>
['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']
Length: 5, dtype: str

In [41]:
filtered_embeddings = model.encode(df_non_legacy['log_message'].tolist())
filtered_embeddings[:2]

array([[-1.02939658e-01,  3.35459076e-02, -2.20260452e-02,
         1.55100238e-03, -9.86919273e-03, -1.78956255e-01,
        -6.34410605e-02, -6.01761863e-02,  2.81108301e-02,
         5.99620342e-02, -1.72619056e-02,  1.43371313e-03,
        -1.49560124e-01,  3.15283332e-03, -5.66031076e-02,
         2.71685813e-02, -1.49890380e-02, -3.54037471e-02,
        -3.62936407e-02, -1.45410290e-02, -5.61491679e-03,
         8.75538141e-02,  4.55120839e-02,  2.50964109e-02,
         1.00187147e-02,  1.24267694e-02, -1.39923632e-01,
         7.68696144e-02,  3.14095356e-02, -4.15251637e-03,
         4.36902493e-02,  1.71250291e-02, -8.00951049e-02,
         5.74006177e-02,  1.89091284e-02,  8.55261385e-02,
         3.96398529e-02, -1.34371802e-01, -1.44369516e-03,
         3.06704547e-03,  1.76854089e-01,  4.44884971e-03,
        -1.69275030e-02,  2.24266127e-02, -4.35050204e-02,
         6.09026849e-03, -9.98166855e-03, -6.23973049e-02,
         1.07372068e-02, -6.04897272e-03, -7.14660585e-0

In [44]:
X = filtered_embeddings
y = df_non_legacy['target_label']

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

                     precision    recall  f1-score   support

     Critical Error       0.97      1.00      0.98        29
              Error       1.00      0.97      0.98        29
        HTTP Status       1.00      1.00      1.00       196
     Resource Usage       1.00      1.00      1.00        29
     Security Alert       1.00      1.00      1.00        81
System Notification       1.00      1.00      1.00        86
        User Action       1.00      1.00      1.00        11

           accuracy                           1.00       461
          macro avg       1.00      1.00      1.00       461
       weighted avg       1.00      1.00      1.00       461



In [45]:
import joblib
joblib.dump(clf, '../models/log_classifier.joblib')

['models/log_classifier.joblib']